In [110]:
import numpy as np
import pandas as pd

from pathlib import Path

from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    Matern,
    WhiteKernel,
)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline

In [111]:
DATA_DIR = Path("./initial_data")
HISTORY_DIR = Path("./history")

N_CANDIDATES = 100_000
SEED = 42

# Set to True if the challenge is a maximisation problem
MAXIMISE = False

In [112]:
def load_function(function_id, data_dir=DATA_DIR):
    path = data_dir / f"function_{function_id}"

    X = np.load(path / "initial_inputs.npy")
    y = np.load(path / "initial_outputs.npy")

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)

    if len(X) != len(y):
        raise ValueError(
            "X and y contain different numbers of observations."
        )

    if np.any(X < 0) or np.any(X > 1):
        raise ValueError(
            "Inputs outside [0, 1]."
        )

    return X, y

In [113]:
def load_history(function_id, history_dir=HISTORY_DIR):
    path = history_dir / f"function_{function_id}_history.csv"

    if not path.exists():
        return None, None

    df = pd.read_csv(path)

    # Columns beginning with x are input dimensions
    x_columns = [
        column
        for column in df.columns
        if column.startswith("x")
    ]

    if "y" not in df.columns:
        raise ValueError(
            f"No y column found in {path}"
        )

    X_history = df[x_columns].to_numpy(dtype=float)
    y_history = df["y"].to_numpy(dtype=float)

    if np.any(X_history < 0) or np.any(X_history > 1):
        raise ValueError(
            f"History contains inputs outside [0, 1]: {path}"
        )

    return X_history, y_history

In [114]:
def load_all_data(function_id):
    X_initial, y_initial = load_function(function_id)

    X_history, y_history = load_history(function_id)

    if X_history is None:
        return X_initial, y_initial

    X = np.vstack([
        X_initial,
        X_history
    ])

    y = np.concatenate([
        y_initial,
        y_history
    ])

    return X, y

In [115]:
def fit_gp(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    dimension = X.shape[1]

    kernel = (
        ConstantKernel(1.0)
        *
        Matern(
            length_scale=np.ones(dimension),
            nu=2.5,
        )
        +
        WhiteKernel(1e-5)
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        optimizer=None,
        random_state=42,
    )

    model.fit(X_scaled, y)

    return model, scaler


In [116]:
def expected_improvement(X_candidates, model, scaler, y_best):
    X_scaled = scaler.transform(X_candidates)

    mean, std = model.predict(
        X_scaled,
        return_std=True
    )

    std = np.maximum(std, 1e-12)

    improvement = y_best - mean

    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        +
        std * norm.pdf(z)
    )

    return ei

In [117]:
def fit_linear_model(X, y):
    model = make_pipeline(
        StandardScaler(),
        LinearRegression()
    )

    model.fit(X, y)

    return model

In [118]:
def fit_logistic_model(X, y):
    # Define the best 30% of observations as "promising"
    threshold = np.quantile(y, 0.30)

    labels = (y <= threshold).astype(int)

    # Logistic regression requires both classes.
    if len(np.unique(labels)) < 2:
        return None

    # With very small data sets, regularisation is important.
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=SEED,
        )
    )

    model.fit(X, labels)

    return model

In [119]:
def generate_candidates(n_candidates, dimension, rng):
    return rng.uniform(0, 1, size=(n_candidates, dimension))

In [120]:
def propose_next(X, y, maximise=False, n_candidates=N_CANDIDATES, seed=SEED):
    rng = np.random.default_rng(seed)

    # Convert maximisation into minimisation
    target = -y if maximise else y

    # --------------------------------------------------------
    # Gaussian Process
    # --------------------------------------------------------
    gp, scaler = fit_gp(
        X,
        target
    )

    # --------------------------------------------------------
    # Linear model
    # --------------------------------------------------------
    linear_model = fit_linear_model(
        X,
        target
    )

    # --------------------------------------------------------
    # Logistic model
    # --------------------------------------------------------
    logistic_model = fit_logistic_model(
        X,
        target
    )

    # --------------------------------------------------------
    # Candidate points
    # --------------------------------------------------------
    candidates = generate_candidates(
        n_candidates=n_candidates,
        dimension=X.shape[1],
        rng=rng,
    )

    # --------------------------------------------------------
    # GP expected improvement
    # --------------------------------------------------------
    y_best = np.min(target)

    ei = expected_improvement(
        candidates,
        gp,
        scaler,
        y_best,
    )

    # --------------------------------------------------------
    # Linear model prediction
    # --------------------------------------------------------
    linear_prediction = linear_model.predict(
        candidates
    )

    # Convert linear prediction into a useful score.
    # Lower predicted y is better.
    linear_range = (
        np.max(linear_prediction)
        -
        np.min(linear_prediction)
        +
        1e-12
    )

    linear_score = (
        np.max(linear_prediction)
        -
        linear_prediction
    ) / linear_range

    # --------------------------------------------------------
    # Logistic probability of being promising
    # --------------------------------------------------------
    if logistic_model is not None:

        promising_probability = (
            logistic_model.predict_proba(
                candidates
            )[:, 1]
        )

    else:
        # If there is not enough information for logistic
        # regression, don't penalise candidates.
        promising_probability = np.ones(
            len(candidates)
        )

    # Normalise EI
    # --------------------------------------------------------
    ei_range = (
        np.max(ei)
        -
        np.min(ei)
        +
        1e-12
    )

    ei_score = (
        ei - np.min(ei)
    ) / ei_range

    # --------------------------------------------------------
    # Combined acquisition score
    # --------------------------------------------------------
    score = (
        0.70 * ei_score
        +
        0.20 * linear_score
        +
        0.10 * promising_probability
    )

    best_idx = np.argmax(score)

    return (
        candidates[best_idx],
        gp,
        scaler,
        linear_model,
        logistic_model,
        score[best_idx],
    )

In [121]:
def format_query(x):
    return "-".join(
        f"{value:.6f}"
        for value in x
    )

In [122]:
for i in range(1, 9):
    try:
        X, y = load_all_data(i)

    except FileNotFoundError:
        print(f"\nFunction {i}: data not found.")
        continue

    x_next, gp, scaler, linear_model, logistic_model, score = (
        propose_next(X, y, maximise=MAXIMISE)
    )

    if MAXIMISE:
        best_idx = y.argmax()
    else:
        best_idx = y.argmin()

    best_x = X[best_idx]
    best_y = y[best_idx]

    query = format_query(x_next)

    print("\n====================================")
    print(f"FUNCTION {i}")
    print("====================================")
    print("Observations:        ", X.shape[0])
    print("Dimensions:          ", X.shape[1])
    print("Current best x:      ", best_x)
    print("Current best y:      ", f"{best_y:.6f}")
    print("Proposed x:")
    print(f"        {x_next}")
    print("Acquisition score:   ", f"{score:.6f}")
    print("Query:")
    print(f"        {query}")


FUNCTION 1
Observations:         11
Dimensions:           2
Current best x:       [0.65011406 0.68152635]
Current best y:       -0.003606
Proposed x:
        [0.58488431 0.59896299]
Acquisition score:    0.850978
Query:
        0.584884-0.598963

FUNCTION 2
Observations:         11
Dimensions:           2
Current best x:       [0.142699 0.349005]
Current best y:       -0.065624
Proposed x:
        [0.16418235 0.17815022]
Acquisition score:    0.923719
Query:
        0.164182-0.178150

FUNCTION 3
Observations:         16
Dimensions:           3
Current best x:       [0.151837 0.439991 0.990882]
Current best y:       -0.398926
Proposed x:
        [0.14363206 0.51663605 0.99737344]
Acquisition score:    0.950674
Query:
        0.143632-0.516636-0.997373

FUNCTION 4
Observations:         31
Dimensions:           4
Current best x:       [0.94838936 0.89451301 0.85163782 0.55219629]
Current best y:       -32.625660
Proposed x:
        [0.96879335 0.9316155  0.85469218 0.69449735]
Acquisitio